# CEG-WM L3 Content Chain 端到端验证 Notebook

**L3 算法实现层升级**：本 Notebook 验证真实 Content Chain 水印机制（UnifiedContentExtractor）

## A. 运行前准备：上传项目 ZIP

In [ ]:
# 克隆仓库到 Colab 环境
import os

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "Content_Chain"  # 包含 supported_models 声明的分支
REPO_NAME = "CEG-WM"
WORK_DIR = f"/content/{REPO_NAME}"

# 先切换到安全的目录（/content）
os.chdir("/content")
print(f"切换到临时目录: {os.getcwd()}")

# 如果目录已存在，先删除
if os.path.exists(WORK_DIR):
  print(f"清空已存在的目录: {WORK_DIR}")
  import shutil
  shutil.rmtree(WORK_DIR)
  print("✅ 已清空")

# 克隆仓库（从 Content_Chain 分支）
print(f"\n克隆仓库: {REPO_URL} (分支: {REPO_BRANCH})")
!git clone -b {REPO_BRANCH} {REPO_URL}

# 进入工作目录
os.chdir(WORK_DIR)
print(f"\n当前工作目录: {os.getcwd()}")
print(f"✅ 仓库克隆完成: {WORK_DIR} (源于分支: {REPO_BRANCH})")

## B. 安装依赖与环境固定

In [ ]:
# Cell B: 安装依赖
from pathlib import Path

print("=" * 60)
print("[B] 安装依赖包")
print("=" * 60)

# 检查 requirements.txt 是否存在
req_file = Path("requirements.txt")
if not req_file.exists():
    raise RuntimeError("未找到 requirements.txt 文件")

print("\n正在安装依赖（这可能需要几分钟）...")

# 安装核心依赖（使用仓库锁定版本）
!pip install -q --upgrade pip
!pip install -q torch==2.10.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers==0.32.0 transformers==4.45.2 accelerate==1.12.0
!pip install -q safetensors==0.7.0 pyyaml==6.0.2 pillow==11.2.1
!pip install -q numpy scipy pytest pydantic
!pip install -q huggingface_hub

print("\n✅ 依赖安装完成")

# 验证关键包版本
print("\n关键包版本:")
import torch
print(f"  - PyTorch: {torch.__version__}")
print(f"  - CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - CUDA 版本: {torch.version.cuda}")
    print(f"  - GPU 设备: {torch.cuda.get_device_name(0)}")

try:
    import diffusers
    print(f"  - Diffusers: {diffusers.__version__}")
except ImportError:
    print("  - Diffusers: ❌ 未安装")

try:
    import transformers
    print(f"  - Transformers: {transformers.__version__}")
except ImportError:
    print("  - Transformers: ❌ 未安装")

try:
    import safetensors
    print(f"  - Safetensors: {safetensors.__version__}")
except ImportError:
    print("  - Safetensors: ❌ 未安装")

## C. 配置 Hugging Face 凭据（可选）

In [ ]:
# 配置 HF_TONKEN

v = os.environ.get("HF_TOKEN")
print("HF_TOKEN exists:", v is not None)
print("HF_TOKEN length:", 0 if v is None else len(v))
print("HF_TOKEN head:", "" if v is None else v[:4] + "..." + v[-4:])

# 下载 Stable Diffusion 模型
print("="*80)
print("下载 Stable Diffusion 模型")
print("="*80)

from diffusers import DiffusionPipeline
from huggingface_hub import scan_cache_dir
import torch

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

# 检查模型是否已缓存
def checkModelCached(modelId):
    try:
        cacheInfo = scan_cache_dir()
        for repo in cacheInfo.repos:
            if modelId in repo.repo_id:
                return True
        return False
    except:
        return False

if checkModelCached(MODEL_ID):
    print(f"模型已缓存: {MODEL_ID}")
    print("跳过下载")
else:
    print(f"模型未缓存 开始下载: {MODEL_ID}")
    print("这可能需要几分钟时间...")

    try:
        # 下载模型但不加载到内存
        _ = DiffusionPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        print("模型下载完成")
    except Exception as e:
        print(f"模型下载失败: {e}")
        print("尝试继续执行...")

# 清理内存
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("模型准备完成")   

## D. 仓库自检

In [ ]:
# Cell D: 仓库自检
import sys

print("=" * 60)
print("[D] 仓库自检")
print("=" * 60)

# 语法检查
print("\n[1/3] 编译检查 main/ 模块...")
!python -m compileall main/ 2>&1 | tail -5

# 导入检查
print("\n[2/3] 测试导入核心模块...")
try:
    sys.path.insert(0, os.getcwd())
    import main
    from main.core import config_loader, digests, records_io
    from main.diffusion.sd3 import pipeline_factory
    print("✅ 核心模块导入成功")
except Exception as e:
    print(f"❌ 导入失败: {e}")
    raise

# 打印仓库版本信息
print("\n[3/3] 仓库冻结契约版本:")
try:
    from main.core.contracts import load_frozen_contracts
    contracts = load_frozen_contracts("configs/frozen_contracts.yaml")
    print(f"  - contract_version: {contracts.contract_version}")
    print(f"  - contract_digest: {contracts.contract_digest[:16]}...")
    print(f"  - contract_bound_digest: {contracts.contract_bound_digest[:16]}...")
except Exception as e:
    print(f"⚠️  无法读取冻结契约: {e}")

print("\n✅ 仓库自检完成")

## E. 准备运行配置

In [ ]:
# Cell E: 定义本地辅助函数
import yaml
from pathlib import Path
from typing import Dict, Any


def torch_cuda_available() -> bool:
    """检查 CUDA 是否可用。"""
    try:
        import torch
        return torch.cuda.is_available()
    except:
        return False


def generate_embed_config(
    output_path: str,
    model_id: str = "stabilityai/stable-diffusion-3.5-medium",
    model_source: str = "hf",
    hf_revision: str = "main",
    inference_prompt: str = "a serene mountain landscape with a lake",
    inference_num_steps: int = 20,
    inference_guidance_scale: float = 4.5,
    seed: int = 42,
    enable_content: bool = True,
    enable_geometry: bool = False,
    enable_paper_faithfulness: bool = True,
    enable_pipeline_inference: bool = True,
    enable_trajectory_tap: bool = True,
    enable_real_inference: bool = True,
    use_fp16: bool = True,
) -> Dict[str, Any]:
    """
    生成 embed 运行配置文件（L3 真实 Content Chain 实现）。

    Generate runtime configuration for embed stage with L3 UnifiedContentExtractor.

    Args:
        output_path: Output YAML file path.
        model_id: Hugging Face model ID.
        model_source: Model source ('hf' or 'local').
        hf_revision: Hugging Face revision (branch/tag/commit hash).
        inference_prompt: Text prompt for inference.
        inference_num_steps: Number of diffusion steps (20-28 recommended for SD3).
        inference_guidance_scale: Guidance scale (4.5-7.0 recommended for SD3).
        seed: Random seed for reproducibility.
        enable_content: Enable content chain watermarking.
        enable_geometry: Enable geometry chain watermarking.
        enable_paper_faithfulness: Enable paper faithfulness alignment check.
        enable_pipeline_inference: Enable P1 (pipeline fingerprint).
        enable_trajectory_tap: Enable P2 (trajectory digest).
        enable_real_inference: Enable real SD3 inference (non-placeholder).
        use_fp16: Use float16 for memory efficiency in Colab.

    Returns:
        Configuration dict.
    """
    if not isinstance(output_path, str) or not output_path:
        raise ValueError("output_path must be non-empty string")
    if not isinstance(model_id, str) or not model_id:
        raise ValueError("model_id must be non-empty string")
    if model_source not in ("hf", "local"):
        raise ValueError("model_source must be 'hf' or 'local'")
    if inference_num_steps < 1 or inference_num_steps > 100:
        raise ValueError("inference_num_steps must be in range [1, 100]")
    if inference_guidance_scale < 0:
        raise ValueError("inference_guidance_scale must be non-negative")

    # 当启用 paper_faithfulness 时，必须强制启用所有必要的真实实现参数
    actual_inference_enabled = enable_pipeline_inference and enable_real_inference
    actual_tap_enabled = enable_trajectory_tap
    if enable_paper_faithfulness:
        actual_inference_enabled = True
        actual_tap_enabled = True

    runtime_config = {
        "policy_path": "content_only",
        "target_fpr": 0.01,
        "device": "cuda" if torch_cuda_available() else "cpu",
        "model_id": model_id,
        "hf_revision": hf_revision,
        
        # 推理配置（真实闭合验证）
        "inference_enabled": actual_inference_enabled,
        "inference_prompt": inference_prompt,
        "inference_num_steps": inference_num_steps,
        "inference_guidance_scale": inference_guidance_scale,
        "inference_height": 512,
        "inference_width": 512,
        
        # 轨迹采样配置（Trajectory Tap）
        "trajectory_tap": {
            "enabled": actual_tap_enabled,
            "tensor_types": ["latent"],
            "module_paths": ["transformer"],
            "stats_precision_digits": 6,
        },
        
        # Paper Faithfulness 配置
        "paper_faithfulness": {
            "enabled": enable_paper_faithfulness,
            "alignment_check": True,
            "check_pipeline_fingerprint": True,
            "check_trajectory_digest": True,
        },
        
        # 水印子空间配置
        "watermark": {
            "subspace": {
                "timestep_start": 0,
                "timestep_end": inference_num_steps - 1,
                "trajectory_step_stride": 1,
                "sample_count": inference_num_steps,
                "feature_dim": 128,
            },
            "lf": {
                "enabled": True,
                "variance": 1.5,
                "ecc_sparsity": 3,
                "message_length": 64,
            },
            "hf": {
                "enabled": False,
                "tail_truncation_ratio": 0.1,
                "tail_truncation_mode": "top_k_per_latent",
                "tau": 2.0,
            },
        },
        
        # 检测配置（Embed 模式）
        "detect": {
            "content": {"enabled": False},  # Embed 模式：检测禁用
            "geometry": {"enabled": False},
        },
        
        # 内容链配置（L3 关键）
        "enable_mask": True,  # 启用语义掩码提取（L3 真实水印必需）
        
        # 实现标识（L3 UnifiedContentExtractor）
        "impl": {
            "content_extractor_id": "unified_content_extractor_v1",  # L3 真实实现（Embed + Detect 双模式）
            "geometry_extractor_id": "geometry_baseline_noop_v1",
            "fusion_rule_id": "fusion_baseline_identity_v1",
            "subspace_planner_id": "subspace_planner_v1",
            "sync_module_id": "geometry_sync_baseline_v1",
        },
    }

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w") as f:
        yaml.dump(runtime_config, f, default_flow_style=False, allow_unicode=True)

    return runtime_config


def generate_detect_config(
    output_path: str,
    base_embed_config_path: str,
) -> Dict[str, Any]:
    """
    生成 detect 配置（基于 embed 配置，启用 detect.content.enabled）。
    
    Generate detect configuration based on embed config with content detection enabled.
    
    Args:
        output_path: Output detect config YAML path.
        base_embed_config_path: Embed config YAML path to copy from.
        
    Returns:
        Detect configuration dict.
    """
    # 读取 embed 配置
    with open(base_embed_config_path, "r") as f:
        detect_cfg = yaml.safe_load(f)
    
    # 修改为 detect 模式
    if "detect" not in detect_cfg:
        detect_cfg["detect"] = {}
    if "content" not in detect_cfg["detect"]:
        detect_cfg["detect"]["content"] = {}
    
    detect_cfg["detect"]["content"]["enabled"] = True  # 启用内容检测
    
    # 保存 detect 配置
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w") as f:
        yaml.dump(detect_cfg, f, default_flow_style=False, allow_unicode=True)
    
    return detect_cfg


def print_config_summary(cfg: Dict[str, Any], config_path: str) -> None:
    """打印配置摘要，突出 L3 真实实现参数。"""
    print("\n" + "=" * 80)
    print("[CONFIG] L3 Content Chain 配置已生成")
    print("=" * 80)

    print(f"\n[文件路径]")
    print(f"  {config_path}")
    print(f"  大小: {Path(config_path).stat().st_size} 字节")

    print(f"\n[模型参数]")
    print(f"  - model_id: {cfg.get('model_id')}")
    print(f"  - device: {cfg.get('device', 'cpu')}")

    print(f"\n[推理参数]")
    print(f"  - inference_prompt: {cfg.get('inference_prompt')}")
    print(f"  - inference_num_steps: {cfg.get('inference_num_steps')}")
    print(f"  - inference_guidance_scale: {cfg.get('inference_guidance_scale')}")

    print(f"\n[L3 Content Chain 启用]")
    impl = cfg.get("impl", {})
    extractor_id = impl.get("content_extractor_id", "N/A")
    enable_mask = cfg.get("enable_mask", False)
    detect_enabled = cfg.get("detect", {}).get("content", {}).get("enabled", False)
    
    print(f"  ✅ content_extractor_id: {extractor_id}")
    print(f"  ✅ enable_mask: {enable_mask}")
    print(f"  {'✅' if detect_enabled else '  '} detect.content.enabled: {detect_enabled} {'[Detect Mode]' if detect_enabled else '[Embed Mode]'}")
    
    print(f"\n[Paper Faithfulness]")
    pf = cfg.get("paper_faithfulness", {})
    if pf.get("enabled"):
        print(f"  ✅ Paper Faithfulness: 启用")
        inf_en = cfg.get("inference_enabled")
        traj_en = cfg.get("trajectory_tap", {}).get("enabled")
        print(f"    [P1] Pipeline 推理: {'✅ 启用' if inf_en else '❌ 禁用'}")
        print(f"    [P2] 轨迹采样: {'✅ 启用' if traj_en else '❌ 禁用'}")
    else:
        print(f"  ❌ Paper Faithfulness: 禁用")

    print(f"\n✅ 配置准备完成\n")

print("✅ 辅助函数已定义（L3 版本）")

## F. 定义运行路径 + 生成配置（真实实现版本）

In [ ]:
# Cell F: 定义运行路径 + 生成配置（L3 真实实现版本）
import os
from pathlib import Path

print("=" * 60)
print("[F] 准备运行环境（L3 Content Chain + 真实 Embed 实现）")
print("=" * 60)

# 定义运行相关的路径
RUN_ROOT = Path("/content/CEG-WM/runs/sd3_l3_content_chain_run")
LOGS_DIR = Path("/content/CEG-WM/logs")
TEMP_CONFIG_EMBED = Path("/content/temp_runtime_embed.yaml")
TEMP_CONFIG_DETECT = Path("/content/temp_runtime_detect.yaml")

LOGS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n[1/3] 运行路径配置:")
print(f"  - RUN_ROOT: {RUN_ROOT}")
print(f"  - LOGS_DIR: {LOGS_DIR}")
print(f"  - TEMP_CONFIG_EMBED: {TEMP_CONFIG_EMBED}")
print(f"  - TEMP_CONFIG_DETECT: {TEMP_CONFIG_DETECT}")

# 生成 embed 配置（L3 真实实现）
print(f"\n[2/3] 生成 Embed 配置 (L3 UnifiedContentExtractor)...")
print("ℹ️  使用优化配置：25 步推理 + float16 + L3 Content Chain\n")

cfg_embed = generate_embed_config(
    output_path=str(TEMP_CONFIG_EMBED),
    model_id="stabilityai/stable-diffusion-3.5-medium",
    model_source="hf",
    hf_revision="main",
    inference_prompt="a beautiful landscape with mountains and lake",
    inference_num_steps=25,  # 25 步用于稳定的轨迹采样
    inference_guidance_scale=4.5,
    seed=42,
    enable_content=True,
    enable_geometry=False,
    enable_paper_faithfulness=True,  # 启用 paper faithfulness
    enable_pipeline_inference=True,   # P1: Pipeline fingerprint
    enable_trajectory_tap=True,       # P2: Trajectory digest
    enable_real_inference=True,       # 强制启用真实推理
    use_fp16=True,                    # 使用 float16 节省内存
)

print_config_summary(cfg_embed, str(TEMP_CONFIG_EMBED))

# 生成 detect 配置（基于 embed，启用 detect.content.enabled）
print(f"[3/3] 生成 Detect 配置（启用内容检测）...")
cfg_detect = generate_detect_config(
    output_path=str(TEMP_CONFIG_DETECT),
    base_embed_config_path=str(TEMP_CONFIG_EMBED),
)
print(f"  ✅ Detect 配置已生成: {TEMP_CONFIG_DETECT}")
print(f"  ✅ detect.content.enabled: True (Detect Mode)\n")

# 定义全局变量供后续 cell 使用
SEED = 42
INFERENCE_NUM_STEPS = 25
INFERENCE_GUIDANCE_SCALE = 4.5
INFERENCE_PROMPT = "a beautiful landscape with mountains and lake"
ENABLE_PAPER_FAITHFULNESS = True
ENABLE_REAL_INFERENCE = True
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

print("[配置参数确认]")
print(f"  ✓ SEED: {SEED}")
print(f"  ✓ INFERENCE_NUM_STEPS: {INFERENCE_NUM_STEPS}")
print(f"  ✓ INFERENCE_GUIDANCE_SCALE: {INFERENCE_GUIDANCE_SCALE}")
print(f"  ✓ ENABLE_PAPER_FAITHFULNESS: {ENABLE_PAPER_FAITHFULNESS}")
print(f"  ✓ ENABLE_REAL_INFERENCE: {ENABLE_REAL_INFERENCE}")
print(f"\n✅ L3 Content Chain 环境准备完成\n")

## G. 执行 Embed 并验证对齐

In [ ]:
# Cell G: 执行 Embed 并验证对齐
import subprocess
import json
import shutil
from pathlib import Path
import yaml
import sys
import importlib

print("=" * 80)
print("[G] 执行 Embed 并验证对齐（真实推理 + P2 真实生成）")
print("=" * 80)

# === [1/3] 清空模块缓存 ===
print("\n[1/3] 清空 Python 模块缓存（强制重新加载最新代码）")
modules_to_clear = [m for m in sys.modules.keys() if m.startswith('main')]
for m in modules_to_clear:
    del sys.modules[m]
print(f"✅ 已清空 {len(modules_to_clear)} 个 main.* 模块")

# === [2/3] 配置预检查 ===
print("\n[2/3] 配置预检查（确保推理和 TAP 已启用）")
config_path = Path(TEMP_CONFIG)
with open(config_path, "r") as f:
    config_content = yaml.safe_load(f)

modified = False
required_checks = [
    ('inference_enabled', 'inference_enabled'),
    ('pipeline_build_enabled', 'pipeline_build_enabled'),
    ('trajectory_tap.enabled', lambda c: c.get('trajectory_tap', {}).get('enabled')),
    ('paper_faithfulness.enabled', lambda c: c.get('paper_faithfulness', {}).get('enabled')),
]

for check_name, check_path in required_checks:
    if callable(check_path):
        current = check_path(config_content)
    else:
        current = config_content.get(check_path)
    
    if not current:
        print(f"  ⚠️  {check_name} 为 False，正在启用...")
        if '.' in check_path:
            parts = check_path.split('.')
            obj = config_content
            for part in parts[:-1]:
                if part not in obj:
                    obj[part] = {}
                obj = obj[part]
            obj[parts[-1]] = True
        else:
            config_content[check_path] = True
        modified = True
    else:
        print(f"  ✅ {check_name}: True")

if modified:
    with open(config_path, "w") as f:
        yaml.dump(config_content, f, default_flow_style=False, allow_unicode=True)
    print("\n✅ 配置文件已更新并保存")

# === [3/3] 执行 Embed ===
print("\n[3/3] 执行 Embed（期望 P2 真实生成）")

# 清空 run_root 目录
if Path(RUN_ROOT).exists():
    print(f"清空 run_root: {RUN_ROOT}")
    shutil.rmtree(RUN_ROOT)
    print("✅ 已清空")

Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)

embed_log = Path(LOGS_DIR) / "embed.log"
cmd = [
    "python", "-m", "main.cli.run_embed",
    "--out", str(RUN_ROOT),
    "--config", str(TEMP_CONFIG)
]

print(f"\n执行命令: {' '.join(cmd)}")
print(f"日志文件: {embed_log}\n")

with open(embed_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n❌ Embed 执行失败，退出码: {proc.returncode}")
    raise RuntimeError(f"Embed 执行失败: 退出码 {proc.returncode}")

print("\n✅ Embed 执行成功")

# === [4/4] 诊断 P1/P2 生成状态 + 深度调试 ===
print("\n[4/4] 诊断 Paper Faithfulness 证据状态（含深度调试）")

# 定位 embed_record.json
records_dir = Path(RUN_ROOT) / "records"
embed_record_files = list(records_dir.glob("embed_*.json"))

if not embed_record_files:
    print("❌ 未找到 embed_record.json")
    raise FileNotFoundError("未找到 embed_record.json")

embed_record_path = embed_record_files[0]
print(f"\n✅ 找到 embed record: {embed_record_path.name}")

# 读取 embed_record
with open(embed_record_path, "r") as f:
    embed_record = json.load(f)

# === 【关键】检查 run_closure.json 中的推理状态 ===
print("\n[深度调试] 检查推理执行状态...")
artifacts_dir = Path(RUN_ROOT) / "artifacts"
run_closure_path = artifacts_dir / "run_closure.json" if artifacts_dir.exists() else None

inference_status = "unknown"
inference_error = None
if run_closure_path and run_closure_path.exists():
    with open(run_closure_path, "r") as f:
        run_closure = json.load(f)
    inference_status = run_closure.get("inference_status", "unknown")
    inference_error = run_closure.get("inference_error")
    print(f"  - inference_status (from run_closure.json): {inference_status}")
    if inference_error:
        print(f"  - inference_error: {inference_error}")
else:
    # 从 embed_record 中读取
    inference_status = embed_record.get("inference_status", "unknown")
    inference_error = embed_record.get("inference_error")
    print(f"  - inference_status (from embed_record): {inference_status}")
    if inference_error:
        print(f"  - inference_error: {inference_error}")

# 检查关键字段
print("\n[关键摘要检查]")
content_evidence = embed_record.get("content_evidence", {})

pfp_digest = content_evidence.get("pipeline_fingerprint_digest")
# P2 在 trajectory_evidence 内部
traj_evidence = content_evidence.get("trajectory_evidence", {})
traj_digest = traj_evidence.get("trajectory_digest") if isinstance(traj_evidence, dict) else None
alignment_report = content_evidence.get("alignment_report", {})
align_status = alignment_report.get("overall_status", "UNKNOWN")

print(f"  - P1 (pipeline_fingerprint_digest): {'✅ 存在' if pfp_digest else '❌ 缺失'}")
print(f"  - P2 (trajectory_digest): {'✅ 真实生成 ✨' if traj_digest else '❌ 缺失'}")
print(f"  - 对齐状态 (overall_status): {align_status}")

# 检测执行模式
print("\n[执行模式检测]")
is_placeholder_mode = False
try:
    with open(embed_log, "r") as f:
        log_content = f.read()
        is_placeholder_mode = "[Embed] Generating embed record (placeholder)" in log_content
        has_tap_logs = "[Trajectory-TAP]" in log_content
except:
    pass

print(f"  - 执行模式: {'Placeholder' if is_placeholder_mode else '真实推理'}")
print(f"  - 是否执行了推理: {'❌ 否（缺少 [Trajectory-TAP] 日志）' if not has_tap_logs else '✅ 是'}")

# 分析对齐检查详情
print("\n[对齐检查详情]")
checks = alignment_report.get("checks", [])
if checks:
    for check in checks:
        check_name = check.get("check_name", "unknown")
        check_result = check.get("result", "UNKNOWN")
        status_icon = "✅" if check_result == "PASS" else ("❌" if check_result == "FAIL" else "⚠️")
        print(f"  {status_icon} {check_name}: {check_result}")
        if check_result == "FAIL":
            failure_msg = check.get("failure_message", "")
            if failure_msg:
                print(f"     └─ 原因: {failure_msg}")

# P2 生成根因分析
print("\n[P2 生成分析]")
trajectory_evidence = content_evidence.get("trajectory_evidence", {})
if trajectory_evidence:
    traj_status = trajectory_evidence.get("status", "unknown")
    traj_absent_reason = trajectory_evidence.get("trajectory_absent_reason")
    print(f"  - Trajectory TAP 状态: {traj_status}")
    if traj_absent_reason:
        print(f"  - 缺失原因: {traj_absent_reason}")
        
        # 根因映射
        if traj_absent_reason == "inference_failed":
            print(f"  - 根因: 推理失败（inference_error: {inference_error}）")
            print(f"  - 修复建议: 检查 run_closure.json 中的完整错误信息")
        elif traj_absent_reason == "inference_disabled":
            print(f"  - 根因: inference_enabled=false")
        elif traj_absent_reason == "tap_disabled":
            print(f"  - 根因: trajectory_tap.enabled=false")
        elif traj_absent_reason == "unsupported_pipeline":
            print(f"  - 根因: pipeline 不支持 callback_on_step_end")

# 最终诊断总结
print("\n" + "=" * 80)
print("[诊断总结]")
print("=" * 80)

if traj_digest:
    print(f"\n✅ P2 已真实生成")
    if align_status == "PASS":
        print(f"✅ 对齐状态: PASS")
        print(f"\n🎉 论文证据完整！P1 + P2 齐全，对齐检查通过")
    else:
        print(f"⚠️ 对齐状态: {align_status}")
        print(f"\n📋 检查失败项：")
        for check in checks:
            if check.get("result") == "FAIL":
                print(f"   - {check.get('check_name')}: {check.get('failure_message', 'N/A')}")
else:
    print(f"\n❌ P2 未真实生成")
    print(f"\n📋 根本原因分析:")
    if inference_status != "ok":
        print(f"   根因: inference_status = '{inference_status}'（期望 'ok'）")
        if inference_error:
            print(f"   错误信息: {inference_error}")
    else:
        print(f"   根因: inference_status = 'ok'，但推理后没有采集到 trajectory_evidence")
        print(f"   可能原因:")
        print(f"     1. pipeline 不支持 callback_on_step_end 接口")
        print(f"     2. 推理中途异常（检查 embed.log）")
        print(f"     3. trajectory_tap.enabled = false")

print(f"\n✅ Embed 记录诊断完成，供后续处理")


In [ ]:
# Cell G-2: 详细的 embed_record 结构检查
import json
from pathlib import Path
import pprint

print("=" * 80)
print("[G-2] 详细的 embed_record 结构检查")
print("=" * 80)

records_dir = Path(RUN_ROOT) / "records"
embed_record_files = list(records_dir.glob("embed_*.json"))

if embed_record_files:
    embed_record_path = embed_record_files[0]
    with open(embed_record_path, "r") as f:
        embed_record = json.load(f)
    
    content_evidence = embed_record.get("content_evidence", {})
    
    print("\n[content_evidence 字段清单]")
    print("-" * 80)
    for key in sorted(content_evidence.keys()):
        value = content_evidence[key]
        if isinstance(value, dict):
            print(f"  - {key}: <dict, {len(value)} fields>")
        elif isinstance(value, list):
            print(f"  - {key}: <list, {len(value)} items>")
        elif isinstance(value, str):
            if len(value) > 50:
                print(f"  - {key}: {value[:50]}...")
            else:
                print(f"  - {key}: {value}")
        else:
            print(f"  - {key}: {type(value).__name__}")
    
    # 检查 trajectory_digest 字段
    print("\n[trajectory_digest 查询]")
    print("-" * 80)
    
    # 尝试多个字段名
    field_names = [
        "trajectory_digest",
        "diffusion_trace_digest", 
        "trace_digest",
        "tap_digest"
    ]
    
    found = False
    for field_name in field_names:
        if field_name in content_evidence:
            value = content_evidence[field_name]
            print(f"  ✅ 找到: {field_name}")
            if isinstance(value, str):
                print(f"     值: {value[:32]}..." if len(value) > 32 else f"     值: {value}")
            else:
                print(f"     类型: {type(value).__name__}")
            found = True
            break
    
    if not found:
        print(f"  ❌ 未找到 trajectory_digest")
        print(f"     已尝试的字段: {field_names}")
    
    # 检查 trajectory_evidence
    print("\n[trajectory_evidence 查询]")
    print("-" * 80)
    
    traj_ev = content_evidence.get("trajectory_evidence")
    if traj_ev:
        print(f"  ✅ trajectory_evidence 存在")
        if isinstance(traj_ev, dict):
            print(f"     - status: {traj_ev.get('status')}")
            print(f"     - 字段数: {len(traj_ev)}")
            if "trajectory_digest" in traj_ev:
                print(f"     - 含有 trajectory_digest: {traj_ev.get('trajectory_digest')[:32]}...")
        else:
            print(f"     类型: {type(traj_ev).__name__}")
    else:
        print(f"  ❌ trajectory_evidence 为 None 或不存在")
    
    # 检查 alignment_report
    print("\n[alignment_report 检查]")
    print("-" * 80)
    
    align_report = content_evidence.get("alignment_report", {})
    if align_report:
        overall_status = align_report.get("overall_status")
        print(f"  - overall_status: {overall_status}")
        
        checks = align_report.get("checks", [])
        print(f"  - checks 数量: {len(checks)}")
        for check in checks:
            check_name = check.get("check_name", "unknown")
            result = check.get("result", "UNKNOWN")
            status_icon = "✅" if result == "PASS" else "❌"
            print(f"    {status_icon} {check_name}: {result}")
    else:
        print(f"  ❌ alignment_report 为空")
    
    print("\n" + "=" * 80)
    print("[总结]")
    print("=" * 80)
    
    # 重新判定 P2 状态
    traj_digest_exists = False
    for field_name in ["trajectory_digest", "diffusion_trace_digest"]:
        if field_name in content_evidence and content_evidence[field_name]:
            traj_digest_exists = True
            print(f"\n✅ P2 确实已生成（在字段 '{field_name}' 中）")
            print(f"   值: {content_evidence[field_name][:32]}...")
            break
    
    if not traj_digest_exists:
        print(f"\n❓ P2 状态不确定，尝试从 trajectory_evidence 中查找")
        if traj_ev and isinstance(traj_ev, dict):
            sub_digest = traj_ev.get("trajectory_digest")
            if sub_digest:
                print(f"   ✅ 在 trajectory_evidence.trajectory_digest 中找到: {sub_digest[:32]}...")
            else:
                print(f"   ❌ trajectory_evidence 中也没有 trajectory_digest")
else:
    print("❌ 未找到 embed_record.json")


## H. 执行 Detect 并生成审计证据

In [ ]:
# Cell H: 执行 Detect + 验证 L3 Content Chain 证据
import subprocess
import json
import shutil
from pathlib import Path
import pandas as pd

print("=" * 80)
print("[H] 执行 Detect（L3 Content Chain 检测模式）")
print("=" * 80)

if 'embed_record_path' not in locals() or embed_record_path is None:
    print("\n⚠️  跳过 detect（embed_record 未找到）")
    print("提示: 请先成功运行 Cell G 以生成 embed_record")
else:
    # === [1/3] 执行 Detect ===
    print("\n" + "=" * 60)
    print("[H-1] 执行 Detect（使用 detect 专用配置）")
    print("=" * 60)
    
    detect_run_root = str(RUN_ROOT) + "_detect"
    
    if Path(detect_run_root).exists():
        print(f"\n清空已存在的 detect run_root 目录: {detect_run_root}")
        shutil.rmtree(detect_run_root)
        print("✅ 已清空")
    
    Path(detect_run_root).mkdir(parents=True, exist_ok=True)
    
    # 使用 detect 专用配置（detect.content.enabled=true）
    cmd = [
        "python", "-m", "main.cli.run_detect",
        "--out", detect_run_root,
        "--config", str(TEMP_CONFIG_DETECT),  # 使用 detect 配置
        "--input", str(embed_record_path)
    ]
    
    print("\n执行命令:")
    print(" ".join(cmd))
    print(f"\n使用配置: {TEMP_CONFIG_DETECT}")
    print("  ✅ detect.content.enabled: true (L3 检测模式)")
    
    detect_log = Path(LOGS_DIR) / "detect.log"
    print(f"\n日志输出: {detect_log}")
    print("\n开始执行...\n")
    
    with open(detect_log, "w") as f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
        
        proc.wait()
    
    if proc.returncode != 0:
        print(f"\n❌ Detect 失败，退出码: {proc.returncode}")
        print("\n最后 50 行日志:")
        try:
            with open(detect_log, "r") as f:
                lines = f.readlines()
                for line in lines[-50:]:
                    print(line, end="")
        except:
            print("无法读取日志")
    else:
        print("\n✅ Detect 执行成功")
        
        detect_records_dir = Path(detect_run_root) / "records"
        detect_record_files = list(detect_records_dir.glob("detect_*.json"))
        
        if detect_record_files:
            detect_record_path = detect_record_files[0]
            print(f"\n找到 detect record: {detect_record_path.name}")
            
            with open(detect_record_path, "r") as f:
                detect_record = json.load(f)
            
            # === [2/3] 验证 L3 Content Chain 证据 ===
            print("\n" + "=" * 60)
            print("[H-2] 验证 L3 Content Chain 检测证据")
            print("=" * 60)
            
            print(f"\n✅ 已加载 detect_record.json，共 {len(detect_record)} 个字段")
            
            content_result = detect_record.get("content_result", {})
            
            print("\n[L3 Content Result 字段]")
            print(f"  - status: {content_result.get('status', 'N/A')}")
            print(f"  - content_score: {content_result.get('score', 'N/A')}")
            print(f"  - content_failure_reason: {content_result.get('content_failure_reason', 'N/A')}")
            
            if content_result.get('status') == 'ok' and content_result.get('score') is not None:
                print(f"\n✅ L3 Content 检测成功，得分: {content_result.get('score')}")
            elif content_result.get('status') in ['failed', 'absent']:
                print(f"\n⚠️  L3 Content 检测未完成: {content_result.get('status')}")
                if content_result.get('content_failure_reason'):
                    print(f"    原因: {content_result.get('content_failure_reason')}")
            
            # === [3/3] 生成 Embed ↔ Detect 对照表 ===
            print("\n" + "=" * 60)
            print("[H-3] Embed ↔ Detect L3 摘要对照表")
            print("=" * 60)
            
            try:
                embed_record_path_val = Path(RUN_ROOT) / "records" / "embed_record.json"
                
                if embed_record_path_val.exists():
                    with open(embed_record_path_val) as f:
                        embed_rec = json.load(f)
                    
                    print("\n[L3 关键字段对照]")
                    print("-" * 80)
                    
                    embed_cr = embed_rec.get("content_result", {})
                    detect_cr = content_result
                    
                    digest_checks = []
                    
                    # mask_digest
                    embed_mask_digest = embed_cr.get("mask_digest")
                    detect_mask_digest = detect_cr.get("mask_digest")
                    mask_match = "✅" if (embed_mask_digest and detect_mask_digest and embed_mask_digest == detect_mask_digest) else "❌"
                    digest_checks.append({
                        "字段": "mask_digest",
                        "Embed": embed_mask_digest[:16] + "..." if embed_mask_digest else "<absent>",
                        "Detect": detect_mask_digest[:16] + "..." if detect_mask_digest else "<absent>",
                        "对齐": mask_match
                    })
                    
                    # plan_digest
                    embed_plan_digest = embed_rec.get("plan_digest")
                    detect_plan_digest = detect_record.get("plan_digest")
                    plan_match = "✅" if (embed_plan_digest and detect_plan_digest and embed_plan_digest == detect_plan_digest) else "❌"
                    digest_checks.append({
                        "字段": "plan_digest",
                        "Embed": embed_plan_digest[:16] + "..." if embed_plan_digest else "<absent>",
                        "Detect": detect_plan_digest[:16] + "..." if detect_plan_digest else "<absent>",
                        "对齐": plan_match
                    })
                    
                    # basis_digest
                    embed_basis_digest = embed_rec.get("basis_digest")
                    detect_basis_digest = detect_record.get("basis_digest")
                    basis_match = "✅" if (embed_basis_digest and detect_basis_digest and embed_basis_digest == detect_basis_digest) else "❌"
                    digest_checks.append({
                        "字段": "basis_digest",
                        "Embed": embed_basis_digest[:16] + "..." if embed_basis_digest else "<absent>",
                        "Detect": detect_basis_digest[:16] + "..." if detect_basis_digest else "<absent>",
                        "对齐": basis_match
                    })
                    
                    df = pd.DataFrame(digest_checks)
                    print("\n" + df.to_string(index=False))
                    
                    align_count = sum(1 for c in digest_checks if c["对齐"] == "✅")
                    total_count = len(digest_checks)
                    
                    print(f"\n[L3 摘要对齐统计] {align_count}/{total_count} 通过")
                    
                    if align_count == total_count:
                        print(f"\n✅ L3 Content Chain 完全对齐：所有摘要一致")
                    else:
                        print(f"\n⚠️  部分未对齐：{total_count - align_count} 个摘要不匹配")
                    
                    l3_alignment_report = {
                        "generated_at": pd.Timestamp.now().isoformat(),
                        "checks": digest_checks,
                        "alignment_summary": {
                            "total": total_count,
                            "passed": align_count,
                            "failed": total_count - align_count,
                            "complete_alignment": align_count == total_count
                        }
                    }
                    
                    l3_report_path = Path(LOGS_DIR) / "l3_content_chain_alignment_report.json"
                    with open(l3_report_path, "w") as f:
                        json.dump(l3_alignment_report, f, indent=2, ensure_ascii=False)
                    
                    print(f"\n✅ L3 对照表已保存: {l3_report_path}")
                    
            except Exception as e:
                print(f"\n❌ 错误: {e}")
                import traceback
                traceback.print_exc()
        else:
            print("\n⚠️  未找到 detect_record.json")

## I. 运行 Pytest 测试

In [ ]:
# Cell I: 运行 Pytest 测试
import subprocess
import os
from pathlib import Path

os.environ["CEG_WM_ENABLE_TORCH_TESTS"] = "1"

print("=" * 60)
print("[I] 运行 Pytest 测试")
print("=" * 60)
print("\n✅ 已启用 opt-in 测试: CEG_WM_ENABLE_TORCH_TESTS=1")

pytest_log = Path(LOGS_DIR) / "pytest.log"

print(f"\n日志输出: {pytest_log}")
print("\n开始执行 pytest（这可能需要几分钟）...\n")

cmd = ["python", "-m", "pytest", "tests/", "-v", "--tb=short"]

with open(pytest_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n⚠️  Pytest 有失败用例，退出码: {proc.returncode}")
    print("\n最后 100 行日志:")
    !tail -100 {pytest_log}
else:
    print("\n✅ Pytest 全部通过")

print("\n=== 测试结果摘要 ===")
!grep -E "passed|failed|skipped|error" {pytest_log} | tail -5 || echo "未找到摘要"


## J. 运行严格审计与冻结签署

In [ ]:
# Cell J: 运行严格审计
import subprocess
import json
from pathlib import Path

print("=" * 60)
print("[J] 运行严格审计 (--strict)")
print("=" * 60)

audits_log = Path(LOGS_DIR) / "audits_strict.log"

print(f"\n日志输出: {audits_log}")
print("\n开始执行严格审计...\n")

cmd = ["python", "scripts/run_all_audits.py", "--strict"]

with open(audits_log, "w") as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    audit_output = []
    for line in proc.stdout:
        print(line, end="")
        f.write(line)
        audit_output.append(line)
    
    proc.wait()

if proc.returncode != 0:
    print(f"\n⚠️  审计脚本退出码: {proc.returncode}")

print("\n=== 审计结果摘要 ===")

try:
    audit_json_str = "".join(audit_output)
    json_start = audit_json_str.find("{")
    if json_start >= 0:
        audit_result = json.loads(audit_json_str[json_start:])
        
        summary = audit_result.get("summary", {})
        freeze_decision = summary.get("FreezeSignoffDecision", "UNKNOWN")
        blocking_reasons = summary.get("BlockingReasons", [])
        counts = summary.get("counts", {})
        
        print(f"\n[决策]")
        print(f"  - FreezeSignoffDecision: {freeze_decision}")
        print(f"  - BlockingReasons: {blocking_reasons if blocking_reasons else '无'}")
        
        print(f"\n[统计]")
        print(f"  - PASS: {counts.get('PASS', 0)}")
        print(f"  - FAIL: {counts.get('FAIL', 0)}")
        print(f"  - BLOCK_fails: {counts.get('BLOCK_fails', 0)}")
        
        if freeze_decision == "ALLOW_FREEZE":
            print(f"\n✅ 审计通过，允许冻结 (ALLOW_FREEZE)")
        else:
            print(f"\n❌ 审计未通过: {freeze_decision}")
            
            if counts.get('BLOCK_fails', 0) > 0:
                results = audit_result.get("results", [])
                for result in results:
                    if result.get("severity") == "BLOCK" and result.get("result") == "FAIL":
                        print(f"\n第一条 BLOCK 失败:")
                        print(f"  - audit_id: {result.get('audit_id', 'N/A')}")
                        print(f"  - rule: {result.get('rule', 'N/A')}")
                        print(f"  - impact: {result.get('impact', 'N/A')}")
                        break
        
        audit_result_path = Path(LOGS_DIR) / "audit_result.json"
        with open(audit_result_path, "w") as f:
            json.dump(audit_result, f, indent=2, ensure_ascii=False)
        print(f"\n审计结果已保存: {audit_result_path}")
        
except Exception as e:
    print(f"\n⚠️  无法解析审计结果 JSON: {e}")
    print("\n审计输出最后 50 行:")
    !tail -50 {audits_log}


## K. 生成对齐验收摘要

In [ ]:
# Cell K: 生成对齐验收摘要
import json
from pathlib import Path
from datetime import datetime, timezone

print("=" * 60)
print("[K] 生成对齐验收摘要")
print("=" * 60)

acceptance_summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "notebook_version": "v1.0",
    "run_root": str(RUN_ROOT),
    "model_id": MODEL_ID,
    "configuration": {
        "seed": SEED,
        "inference_steps": INFERENCE_NUM_STEPS,
        "guidance_scale": INFERENCE_GUIDANCE_SCALE,
        "prompt": INFERENCE_PROMPT,
        "enable_paper_faithfulness": ENABLE_PAPER_FAITHFULNESS
    },
    "embed_status": "unknown",
    "detect_status": "unknown",
    "pytest_status": "unknown",
    "audit_status": "unknown",
    "final_verdict": "UNKNOWN"
}

# Embed 状态
if 'embed_record_path' in locals() and embed_record_path and embed_record_path.exists():
    with open(embed_record_path, "r") as f:
        embed_rec = json.load(f)
    
    content_ev = embed_rec.get("content_evidence", {})
    alignment_status = content_ev.get("alignment_report", {}).get("overall_status", "UNKNOWN")
    
    acceptance_summary["embed_status"] = {
        "success": True,
        "alignment_overall_status": alignment_status,
        "record_path": str(embed_record_path)
    }
else:
    acceptance_summary["embed_status"] = {"success": False}

# Detect 状态
try:
    if 'detect_run_root' in locals():
        detect_records_dir = Path(detect_run_root) / "records"
        detect_rec_files = list(detect_records_dir.glob("detect_*.json"))
        
        if detect_rec_files:
            with open(detect_rec_files[0], "r") as f:
                detect_rec = json.load(f)
            
            pf_status = detect_rec.get("paper_faithfulness", {}).get("status", "UNKNOWN")
            is_watermarked = detect_rec.get("decision", {}).get("is_watermarked")
            
            acceptance_summary["detect_status"] = {
                "success": True,
                "paper_faithfulness_status": pf_status,
                "is_watermarked": is_watermarked,
                "record_path": str(detect_rec_files[0])
            }
        else:
            acceptance_summary["detect_status"] = {"success": False, "reason": "no detect_record found"}
    else:
        acceptance_summary["detect_status"] = {"success": False, "reason": "detect not executed"}
except Exception as e:
    acceptance_summary["detect_status"] = {"success": False, "error": str(e)}

# Pytest 状态
pytest_log = Path(LOGS_DIR) / "pytest.log"
if pytest_log.exists():
    with open(pytest_log, "r") as f:
        pytest_content = f.read()
    
    # 解析 pytest 结果
    import re
    match = re.search(r'(\d+) passed', pytest_content)
    passed = int(match.group(1)) if match else 0
    
    match_failed = re.search(r'(\d+) failed', pytest_content)
    failed = int(match_failed.group(1)) if match_failed else 0
    
    acceptance_summary["pytest_status"] = {
        "passed": passed,
        "failed": failed,
        "all_passed": failed == 0 and passed > 0
    }
else:
    acceptance_summary["pytest_status"] = {"executed": False}

# Audit 状态
audit_log_path = Path(LOGS_DIR) / "audit_result.json"
if audit_log_path.exists():
    with open(audit_log_path, "r") as f:
        audit_result = json.load(f)
    
    # 从 summary 中读取冻结决策
    summary = audit_result.get("summary", {})
    freeze_decision = summary.get("FreezeSignoffDecision", "UNKNOWN")
    blocking_reasons = summary.get("BlockingReasons", [])
    counts = summary.get("counts", {})
    
    # 从 results 数组中统计审计结果
    results = audit_result.get("results", [])
    pass_count = sum(1 for r in results if r.get("result") == "PASS")
    fail_count = sum(1 for r in results if r.get("result") == "FAIL")
    block_fails = sum(1 for r in results if r.get("result") == "FAIL" and r.get("severity") == "BLOCK")
    
    acceptance_summary["audit_status"] = {
        "freeze_decision": freeze_decision,
        "blocking_reasons": blocking_reasons,
        "pass_count": pass_count,
        "fail_count": fail_count,
        "block_fails": block_fails
    }
else:
    acceptance_summary["audit_status"] = {"executed": False}

# 最终判定
conditions_met = []
conditions_failed = []

# 检查 Embed 对齐
if isinstance(acceptance_summary["embed_status"], dict):
    if acceptance_summary["embed_status"].get("success") and \
       acceptance_summary["embed_status"].get("alignment_overall_status") == "PASS":
        conditions_met.append("embed_alignment_pass")
    else:
        conditions_failed.append("embed_alignment_not_pass")

# 检查 Detect Paper Faithfulness
if isinstance(acceptance_summary["detect_status"], dict):
    if acceptance_summary["detect_status"].get("success") and \
       acceptance_summary["detect_status"].get("paper_faithfulness_status") == "ok":
        conditions_met.append("detect_pf_ok")
    else:
        conditions_failed.append("detect_pf_not_ok")

# 检查 Pytest
if isinstance(acceptance_summary["pytest_status"], dict):
    if acceptance_summary["pytest_status"].get("all_passed"):
        conditions_met.append("pytest_all_passed")
    else:
        conditions_failed.append("pytest_not_all_passed")

# 检查 Audit
if isinstance(acceptance_summary["audit_status"], dict):
    if acceptance_summary["audit_status"].get("freeze_decision") == "ALLOW_FREEZE":
        conditions_met.append("audit_allow_freeze")
    else:
        conditions_failed.append("audit_not_allow_freeze")

# 判定
if len(conditions_failed) == 0 and len(conditions_met) >= 3:
    acceptance_summary["final_verdict"] = "ACCEPT"
    acceptance_summary["verdict_reason"] = "所有验收条件满足"
else:
    acceptance_summary["final_verdict"] = "REJECT"
    acceptance_summary["verdict_reason"] = f"未满足条件: {', '.join(conditions_failed)}"

# 保存摘要
summary_path = Path("/content/alignment_acceptance_summary.json")
with open(summary_path, "w") as f:
    json.dump(acceptance_summary, f, indent=2, ensure_ascii=False)

print(f"\n✅ 验收摘要已生成: {summary_path}")

print("\n" + "=" * 60)
print("最终验收判定")
print("=" * 60)
print(f"\n结果: {acceptance_summary['final_verdict']}")
print(f"原因: {acceptance_summary['verdict_reason']}")

if acceptance_summary['final_verdict'] == "ACCEPT":
    print("\n✅ 所有验收条件满足，证据包有效")
else:
    print("\n❌ 存在未满足的验收条件")

print(f"\n验收摘要详情:")
print(json.dumps(acceptance_summary, indent=2, ensure_ascii=False))


## L. 打包并下载证据包

In [ ]:
# Cell L: 打包并下载证据包
import shutil
from pathlib import Path
from google.colab import files

print("=" * 60)
print("[L] 打包证据包")
print("=" * 60)

artifacts_out = Path("/content/artifacts_out")
artifacts_out.mkdir(parents=True, exist_ok=True)

print("\n[1/4] 复制 run_root...")
run_root_copy = artifacts_out / "run_root"
if Path(RUN_ROOT).exists():
    shutil.copytree(RUN_ROOT, run_root_copy, dirs_exist_ok=True)
    print(f"  ✅ {RUN_ROOT} → {run_root_copy}")

try:
    if Path(detect_run_root).exists():
        detect_copy = artifacts_out / "run_root_detect"
        shutil.copytree(detect_run_root, detect_copy, dirs_exist_ok=True)
        print(f"  ✅ {detect_run_root} → {detect_copy}")
except:
    pass

print("\n[2/4] 复制 run_logs...")
logs_copy = artifacts_out / "run_logs"
if Path(LOGS_DIR).exists():
    shutil.copytree(LOGS_DIR, logs_copy, dirs_exist_ok=True)
    print(f"  ✅ {LOGS_DIR} → {logs_copy}")

print("\n[3/4] 复制验收摘要...")
summary_src = Path("/content/alignment_acceptance_summary.json")
if summary_src.exists():
    shutil.copy(summary_src, artifacts_out / "alignment_acceptance_summary.json")
    print(f"  ✅ 验收摘要已复制")

print("\n[4/4] 打包为 ZIP...")
bundle_zip = Path("/content/sd3_paper_faithfulness_run_bundle.zip")
shutil.make_archive(
    str(bundle_zip.with_suffix("")),
    'zip',
    artifacts_out
)

bundle_size_mb = bundle_zip.stat().st_size / (1024 * 1024)
print(f"  ✅ 打包完成: {bundle_zip.name} ({bundle_size_mb:.2f} MB)")

print("\n" + "=" * 60)
print("准备下载...")
print("=" * 60)

print("\n下载文件:")
files.download(str(bundle_zip))

print("\n✅ 下载完成！")
print("\n证据包内容:")
print("  - run_root/: embed 运行完整输出（records + artifacts + logs）")
print("  - run_root_detect/: detect 运行完整输出（如果执行成功）")
print("  - run_logs/: 所有执行日志（embed/detect/pytest/audits）")
print("  - alignment_acceptance_summary.json: 验收摘要")
print("\n请解压后检查验收摘要中的 final_verdict 字段。")


## 完成

### L3 Content Chain 验收标准

检查下载的证据包中的关键报告：

#### 1. L3 Content Chain 对齐验证
查看 `l3_content_chain_alignment_report.json`：
- ✅ `alignment_summary.complete_alignment` == `true`
- ✅ `mask_digest` 在 Embed 和 Detect 之间对齐
- ✅ `plan_digest` 在 Embed 和 Detect 之间对齐
- ✅ `basis_digest` 在 Embed 和 Detect 之间对齐

#### 2. Embed Record 验证
查看 `run_root/records/embed_record.json`：
- ✅ `content_result.status` ∈ {"ok", "failed"}
- ✅ `impl_identity.content_extractor_id` == `"unified_content_extractor_v1"`
- ✅ `enable_mask` == `true`

#### 3. Detect Record 验证
查看 `run_root_detect/records/detect_record.json`:
- ✅ `content_result.status` ∈ {"ok", "absent", "failed"}
- ✅ `detect.content.enabled` == `true`（配置确认）
- ✅ `content_score` 存在（当 status="ok" 时）

### L3 实现说明

#### UnifiedContentExtractor（统一内容链提取器）
- **Embed 模式**（`detect.content.enabled=false`）：
  - 委托给 SemanticMaskProvider提取语义掩码
  - 返回 `mask_digest`、`mask_stats` 等结构证据
  
- **Detect 模式**（`detect.content.enabled=true`）：
  - 委托给 ContentDetector
  - 执行完整 LF/HF 检测
  - 返回 `content_score`（检测分数）

#### 关键配置参数
- `enable_mask: true`：启用语义掩码提取（L3 必需）
- `impl.content_extractor_id: unified_content_extractor_v1`：L3 真实实现
- `detect.content.enabled: false/true`：控制 Embed/Detect 模式切换

### 故障排查

如遇到问题：

1. **模型下载失败**：检查 Cell C 是否正确配置 HF_TOKEN
2. **Embed 失败**：查看 `run_logs/embed.log` 最后 50 行
3. **mask_extraction_no_input**：CPU 模式下无图像输入，属于预期行为（需 GPU 真实推理）
4. **L3 对齐未通过**：检查 content_extractor_id 是否为 unified_content_extractor_v1
5. **Detect status=absent**：检查 detect.content.enabled 是否为 true

### 模型说明

本 Notebook 使用 **Stable Diffusion 3.5 Medium**，采用 **DiT (Diffusion Transformer)** 架构，而非传统 UNet。
模型通过 `diffusers` 库的 `StableDiffusion3Pipeline` 加载。

### L3 升级日志（2026-02-21）

- ✅ 创建 UnifiedContentExtractor（Embed+Detect 双模式）
- ✅ 注册到 content_registry（unified_content_extractor_v1）
- ✅ 更新 default.yaml（enable_mask: true, content_extractor_id）
- ✅ 创建 detect_config.yaml（detect.content.enabled: true）
- ✅ 修复 max_resolution 格式（"2048x2048"）
- ✅ 修复 content_result 序列化（调用 as_dict()）
- ✅ 最小 embed+detect 闭环验证通过